# 🫁 Chest X-Ray Classification: COVID-19, Pneumonia, Tuberculosis, Normal
## EfficientNet B0 → B7 | Plain & CBAM-Enhanced | Deep Learning for Medical Imaging
> **Based on:** *Explanatory classification of CXR images into COVID-19, Pneumonia and Tuberculosis using deep learning and XAI* (Bhandari et al., 2022)
> 
> **Dataset:** [Chest X-Ray (Pneumonia, Covid-19, Tuberculosis)](https://www.kaggle.com/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis)
> 
> **Experiments:**  
> - **Part A** — EfficientNet B0 → B7 (plain, pretrained)  
> - **Part B** — EfficientNet B0 → B7 **+ CBAM** (Channel & Spatial Attention injected into every MBConv block)

---

## 📦 Section 1: Setup & Installation

In [ ]:
# Install required libraries
!pip install timm grad-cam -q

In [ ]:
# ── Core ──────────────────────────────────────────────────
import os, random, time, copy, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── PyTorch ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

# ── EfficientNet via timm ──────────────────────────────────
import timm

# ── Visualization ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

# ── Metrics ───────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize

# ── Grad-CAM ──────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print('✅ All libraries imported successfully!')

In [ ]:
# ── Reproducibility ───────────────────────────────────────
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# ── Device ────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device     : {DEVICE}')
if torch.cuda.is_available():
    print(f'🎮  GPU        : {torch.cuda.get_device_name(0)}')
    print(f'💾  VRAM       : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'🔥  PyTorch    : {torch.__version__}')
print(f'🌱  Seed       : {SEED}')

In [ ]:
# ── Paths ─────────────────────────────────────────────────
DATA_DIR   = '/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
VAL_DIR    = os.path.join(DATA_DIR, 'val')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Global Config ─────────────────────────────────────────
CLASS_NAMES  = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']
NUM_CLASSES  = 4
BATCH_SIZE   = 32   # baseline; overridden per-variant below
EPOCHS       = 20
PATIENCE     = 3
NUM_WORKERS  = 2

# ── EfficientNet variants B0 → B7 ─────────────────────────
# NOTE: B6/B7 use tf_ prefix — only names with pretrained weights available in timm
# batch_size reduced for larger inputs to avoid CUDA OOM on ~15 GB VRAM
EFFICIENTNET_CONFIGS = {
    'B0': {'name': 'efficientnet_b0',    'img_size': 224, 'params': '5.3M',  'batch_size': 32},
    'B1': {'name': 'efficientnet_b1',    'img_size': 240, 'params': '7.8M',  'batch_size': 32},
    'B2': {'name': 'efficientnet_b2',    'img_size': 260, 'params': '9.2M',  'batch_size': 32},
    'B3': {'name': 'efficientnet_b3',    'img_size': 300, 'params': '12M',   'batch_size': 32},
    'B4': {'name': 'efficientnet_b4',    'img_size': 380, 'params': '19M',   'batch_size': 16},
    'B5': {'name': 'efficientnet_b5',    'img_size': 456, 'params': '30M',   'batch_size': 8},
    'B6': {'name': 'tf_efficientnet_b6', 'img_size': 528, 'params': '43M',   'batch_size': 6},
    'B7': {'name': 'tf_efficientnet_b7', 'img_size': 600, 'params': '66M',   'batch_size': 4},
}

# ── ImageNet normalization ─────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Colors ────────────────────────────────────────────────
CLASS_COLORS = {
    'COVID19': '#E74C3C', 'NORMAL': '#2ECC71',
    'PNEUMONIA': '#3498DB', 'TURBERCULOSIS': '#F39C12'
}
MODEL_COLORS = {
    'B0': '#9B59B6', 'B1': '#3498DB', 'B2': '#E74C3C', 'B3': '#2ECC71',
    'B4': '#E67E22', 'B5': '#1ABC9C', 'B6': '#34495E', 'B7': '#C0392B',
}

ALL_VARIANTS = ['B0', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']

print('✅ Configuration set!')
print(f'📁 Data directory  : {DATA_DIR}')
print(f'📂 Output directory: {OUTPUT_DIR}')
print(f'🔢 Variants        : {ALL_VARIANTS}')
for v, cfg in EFFICIENTNET_CONFIGS.items():
    print(f'   {v}: model={cfg["name"]}, img={cfg["img_size"]}, batch={cfg["batch_size"]}')


## 📊 Section 2: Exploratory Data Analysis (EDA)

In [ ]:
# ── Count images per class per split ──────────────────────
def count_images(root_dir):
    counts = {}
    for cls in sorted(os.listdir(root_dir)):
        cls_path = os.path.join(root_dir, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.png','.jpg','.jpeg'))
            ])
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR)
test_counts  = count_images(TEST_DIR)

df_counts = pd.DataFrame({
    'Class' : list(train_counts.keys()),
    'Train' : list(train_counts.values()),
    'Val'   : list(val_counts.values()),
    'Test'  : list(test_counts.values()),
})
df_counts['Total']  = df_counts['Train'] + df_counts['Val'] + df_counts['Test']
df_counts['Train%'] = (df_counts['Train'] / df_counts['Total'] * 100).round(1)

print('='*60)
print('📊 DATASET STATISTICS')
print('='*60)
print(df_counts.to_string(index=False))
print('─'*60)
print(f"{'TOTAL':<15} {df_counts['Train'].sum():<8} {df_counts['Val'].sum():<8} "
      f"{df_counts['Test'].sum():<8} {df_counts['Total'].sum()}")
print('='*60)

In [ ]:
# ── Figure 1: Class Distribution ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📊 Class Distribution Across Splits', fontsize=16, fontweight='bold', y=1.02)

splits = [('Train', train_counts), ('Validation', val_counts), ('Test', test_counts)]
for ax, (split_name, counts) in zip(axes, splits):
    classes = list(counts.keys())
    values  = list(counts.values())
    colors  = [CLASS_COLORS.get(c, '#95A5A6') for c in classes]
    bars = ax.bar(classes, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{split_name} Set\n(Total: {sum(values):,})', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Images', fontsize=11)
    ax.set_xticklabels(classes, rotation=15, ha='right')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylim(0, max(values)*1.15)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 01_class_distribution.png')

In [ ]:
import timm
print(timm.__version__)
print([m for m in timm.list_models('*efficientnet_b6*')])
print([m for m in timm.list_models('*efficientnet_b7*')])

## 🔄 Section 3: Data Processing & DataLoaders

In [ ]:
def get_transforms(img_size, split='train'):
    """
    Data transforms:
    - Train: Resize, HFlip, Rotation, ColorJitter, Normalize
    - Val/Test: Resize, Normalize
    """
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ])


def get_dataloaders(img_size, batch_size=32):
    """Create DataLoaders with WeightedRandomSampler for class imbalance."""
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=get_transforms(img_size, 'train'))
    val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=get_transforms(img_size, 'val'))
    test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=get_transforms(img_size, 'test'))

    targets       = train_dataset.targets
    class_counts  = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    return train_loader, val_loader, test_loader, train_dataset.class_to_idx


print('✅ Transform and DataLoader functions defined!')

## 🧠 Section 4: CBAM — Convolutional Block Attention Module

CBAM (Woo et al., 2018) applies **channel attention** then **spatial attention** sequentially.  
Here we wrap the pretrained EfficientNet backbone by injecting a CBAM block after each  
MBConv stage's pointwise-convolution output, giving the model explicit attention at every scale.

```
Input feature map F
  → Channel Attention  → F' = F ⊗ Mc(F)
  → Spatial Attention  → F'' = F' ⊗ Ms(F')
```

In [ ]:
# ─────────────────────────────────────────────────────────
# CBAM: Channel Attention + Spatial Attention
# Reference: Woo et al., ECCV 2018 — "CBAM: Convolutional Block Attention Module"
# ─────────────────────────────────────────────────────────

class ChannelAttention(nn.Module):
    """
    Squeeze-and-Excitation style channel attention.
    Uses both Average-Pool and Max-Pool branches, then adds them before sigmoid.
    """
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        mid = max(in_channels // reduction_ratio, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        scale   = self.sigmoid(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        return x * scale


class SpatialAttention(nn.Module):
    """
    Spatial attention via channel-wise avg & max pooling then 7×7 conv.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size in (3, 7), 'kernel_size must be 3 or 7'
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        cat_out = torch.cat([avg_out, max_out], dim=1)
        scale   = self.sigmoid(self.conv(cat_out))
        return x * scale


class CBAM(nn.Module):
    """Full CBAM block: Channel Attention → Spatial Attention."""
    def __init__(self, in_channels, reduction_ratio=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction_ratio)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


print('✅ CBAM modules defined (ChannelAttention + SpatialAttention + CBAM)')

## 🏗️ Section 5: Model Architecture (Plain & CBAM-Enhanced)

In [ ]:
# ─────────────────────────────────────────────────────────
# Plain EfficientNet (pretrained, custom head)
# ─────────────────────────────────────────────────────────

def build_model_plain(variant='B0'):
    """
    Standard pretrained EfficientNet with replaced classifier head.
    Dropout(0.3) → Linear(num_classes).
    """
    cfg          = EFFICIENTNET_CONFIGS[variant]
    model        = timm.create_model(cfg['name'], pretrained=True)
    in_features  = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)


# ─────────────────────────────────────────────────────────
# EfficientNet + CBAM wrapper
# Strategy: inject CBAM after each MBConv block group (model.blocks[i])
# ─────────────────────────────────────────────────────────

class CBAMBlock(nn.Module):
    """Wraps an EfficientNet MBConv block and appends CBAM after it."""
    def __init__(self, mbconv_block, out_channels):
        super().__init__()
        self.block = mbconv_block
        self.cbam  = CBAM(out_channels)

    def forward(self, x):
        x = self.block(x)
        x = self.cbam(x)
        return x


def build_model_cbam(variant='B0'):
    """
    EfficientNet + CBAM:
    - Load pretrained EfficientNet via timm
    - Wrap the LAST block in each stage (model.blocks[i][-1]) with CBAM
    - Replace classifier head
    """
    cfg   = EFFICIENTNET_CONFIGS[variant]
    model = timm.create_model(cfg['name'], pretrained=True)

    # Inject CBAM at the end of each block group
    for i, stage in enumerate(model.blocks):
        last_block = stage[-1]
        # Determine output channels of this block
        try:
            out_ch = last_block.conv_pwl.out_channels
        except AttributeError:
            try:
                out_ch = last_block.conv_dw.out_channels
            except AttributeError:
                continue   # skip blocks with non-standard structure

        # Replace the last block in the stage with a CBAM-wrapped version
        stage[-1] = CBAMBlock(last_block, out_ch)

    # Replace classifier head
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)


def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ── Print parameter counts ─────────────────────────────────
print('='*80)
print(f'{"Variant":<12} {"Plain Params":<20} {"CBAM Params":<20} {"CBAM Overhead":<15}')
print('='*80)
for variant in ALL_VARIANTS:
    try:
        mp = build_model_plain(variant)
        mc = build_model_cbam(variant)
        tp, _ = count_parameters(mp)
        tc, _ = count_parameters(mc)
        overhead = tc - tp
        print(f'EfficientNet-{variant:<4} {tp/1e6:.2f}M{"":<14} {tc/1e6:.2f}M{"":<14} +{overhead/1e6:.2f}M')
        del mp, mc
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'EfficientNet-{variant:<4} ERROR: {e}')
print('='*80)

## 🚀 Section 6: Training Pipeline

In [ ]:
def compute_class_weights(train_loader):
    """Inverse-frequency class weights for CrossEntropyLoss."""
    targets = train_loader.dataset.targets
    counts  = np.bincount(targets)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    return torch.FloatTensor(weights).to(DEVICE)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            probs    = torch.softmax(outputs, dim=1)
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct  += preds.eq(labels).sum().item()
            total    += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (running_loss/total, correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(variant, model_type='plain'):
    """
    Train one model variant.
    model_type: 'plain' | 'cbam'
    """
    cfg        = EFFICIENTNET_CONFIGS[variant]
    img_size   = cfg['img_size']
    batch_size = cfg.get('batch_size', BATCH_SIZE)
    tag        = f'{variant}_{model_type}'
    label      = f'EfficientNet-{variant}' + ('+CBAM' if model_type=='cbam' else '')

    print(f'\n{"="*65}')
    print(f'🚀 Training {label}  |  Input: {img_size}×{img_size}  |  Batch: {batch_size}')
    print(f'{"="*65}')

    # Free memory before loading new model
    import gc
    torch.cuda.empty_cache()
    gc.collect()

    train_loader, val_loader, test_loader, _ = get_dataloaders(img_size, batch_size=batch_size)

    model     = build_model_plain(variant) if model_type == 'plain' else build_model_cbam(variant)

    # Enable gradient checkpointing for large variants to reduce VRAM usage
    if variant in ('B5', 'B6', 'B7') and hasattr(model, 'set_grad_checkpointing'):
        model.set_grad_checkpointing(enable=True)
        print(f'  ✅ Gradient checkpointing enabled for {variant}')
    cw        = compute_class_weights(train_loader)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler()

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'lr':[]}
    best_val_acc   = 0.0
    patience_ctr   = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    ckpt_path      = f'{OUTPUT_DIR}/best_{tag}.pth'

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc           = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_acc, _, _, _  = evaluate(model, val_loader, criterion)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['lr'].append(lr)

        elapsed = time.time() - t0
        print(f'  Epoch [{epoch:>2}/{EPOCHS}] '
              f'Train: {tr_acc*100:.2f}% Loss:{tr_loss:.4f} | '
              f'Val: {vl_acc*100:.2f}% Loss:{vl_loss:.4f} | '
              f'LR:{lr:.2e} | {elapsed:.1f}s')

        if vl_acc > best_val_acc:
            best_val_acc   = vl_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, ckpt_path)
            patience_ctr   = 0
            print(f'  ✅ Best saved! Val Acc: {best_val_acc*100:.2f}%')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  ⏹️  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(torch.load(ckpt_path))
    _, test_acc, test_preds, test_labels, test_probs = evaluate(model, test_loader, criterion)
    print(f'\n  🎯 Test Accuracy: {test_acc*100:.2f}%')

    return model, history, test_preds, test_labels, test_probs, test_loader


print('✅ Training functions defined!')

## ▶️ Section 7: Run Training — All Variants × Both Types

In [ ]:
# ─────────────────────────────────────────────────────────────
# PART A: Plain EfficientNet  B0 → B7
# ─────────────────────────────────────────────────────────────
plain_results = {}

import gc

for variant in ALL_VARIANTS:
    # Free VRAM before each model
    torch.cuda.empty_cache()
    gc.collect()
    set_seed()
    try:
        model, history, test_preds, test_labels, test_probs, test_loader = \
            train_model(variant, model_type='plain')
        plain_results[variant] = {
            'model'      : model,
            'history'    : history,
            'test_preds' : test_preds,
            'test_labels': test_labels,
            'test_probs' : test_probs,
            'test_loader': test_loader,
        }
    except Exception as e:
        print(f'❌ EfficientNet-{variant} FAILED: {e}')
    finally:
        torch.cuda.empty_cache()

print('\n✅ Part A (Plain) — all variants done!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# PART B: EfficientNet + CBAM  B0 → B7
# ─────────────────────────────────────────────────────────────
cbam_results = {}

for variant in ALL_VARIANTS:
    # Free VRAM before each model
    torch.cuda.empty_cache()
    gc.collect()
    set_seed()
    try:
        model, history, test_preds, test_labels, test_probs, test_loader = \
            train_model(variant, model_type='cbam')
        cbam_results[variant] = {
            'model'      : model,
            'history'    : history,
            'test_preds' : test_preds,
            'test_labels': test_labels,
            'test_probs' : test_probs,
            'test_loader': test_loader,
        }
    except Exception as e:
        print(f'❌ EfficientNet-{variant}+CBAM FAILED: {e}')
    finally:
        torch.cuda.empty_cache()

print('\n✅ Part B (CBAM) — all variants done!')

## 📐 Section 8: Compute Evaluation Metrics

In [ ]:
def compute_metrics(test_preds, test_labels, test_probs):
    """Compute full metric suite for one model."""
    acc       = accuracy_score(test_labels, test_preds)
    precision = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    recall    = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    f1        = f1_score(test_labels, test_preds, average='macro', zero_division=0)

    # Per-class sensitivity & specificity
    cm = confusion_matrix(test_labels, test_preds)
    per_sens, per_spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        per_sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        per_spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

    sensitivity  = np.mean(per_sens)
    specificity  = np.mean(per_spec)

    # Macro AUC
    y_bin = label_binarize(test_labels, classes=list(range(NUM_CLASSES)))
    try:
        roc_auc = roc_auc_score(y_bin, test_probs, average='macro', multi_class='ovr')
    except Exception:
        roc_auc = 0.0

    return {
        'accuracy'      : acc,
        'precision'     : precision,
        'recall'        : recall,
        'f1'            : f1,
        'sensitivity'   : sensitivity,
        'specificity'   : specificity,
        'auc'           : roc_auc,
        'per_class_sens': per_sens,
        'per_class_spec': per_spec,
        'confusion_matrix': cm,
    }


# Compute for all variants
plain_metrics = {}
cbam_metrics  = {}

for variant in ALL_VARIANTS:
    if variant in plain_results:
        r = plain_results[variant]
        plain_metrics[variant] = compute_metrics(r['test_preds'], r['test_labels'], r['test_probs'])
    if variant in cbam_results:
        r = cbam_results[variant]
        cbam_metrics[variant]  = compute_metrics(r['test_preds'], r['test_labels'], r['test_probs'])

print('✅ Metrics computed for all variants (plain & CBAM)!')

## 📈 Section 9: Training Curves

In [ ]:
# ── Figure 1: Plain EfficientNet training curves ──────────
fig, axes = plt.subplots(8, 2, figsize=(18, 48))
fig.suptitle('📈 Training History — Plain EfficientNet B0 → B7', fontsize=16, fontweight='bold')

for row, variant in enumerate(ALL_VARIANTS):
    if variant not in plain_results:
        continue
    h   = plain_results[variant]['history']
    eps = range(1, len(h['train_loss']) + 1)
    c   = MODEL_COLORS[variant]

    axes[row,0].plot(eps, h['train_loss'], color=c, lw=2, label='Train')
    axes[row,0].plot(eps, h['val_loss'],   color=c, lw=2, ls='--', label='Val')
    axes[row,0].set_title(f'EfficientNet-{variant} — Loss', fontweight='bold')
    axes[row,0].set_xlabel('Epoch'); axes[row,0].set_ylabel('Loss')
    axes[row,0].legend(); axes[row,0].spines[['top','right']].set_visible(False)

    axes[row,1].plot(eps, [x*100 for x in h['train_acc']], color=c, lw=2, label='Train')
    axes[row,1].plot(eps, [x*100 for x in h['val_acc']],   color=c, lw=2, ls='--', label='Val')
    axes[row,1].set_title(f'EfficientNet-{variant} — Accuracy', fontweight='bold')
    axes[row,1].set_xlabel('Epoch'); axes[row,1].set_ylabel('Accuracy (%)')
    axes[row,1].legend(); axes[row,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_train_curves_plain.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_train_curves_plain.png')

In [ ]:
# ── Figure 2: CBAM EfficientNet training curves ───────────
fig, axes = plt.subplots(8, 2, figsize=(18, 48))
fig.suptitle('📈 Training History — EfficientNet+CBAM B0 → B7', fontsize=16, fontweight='bold')

for row, variant in enumerate(ALL_VARIANTS):
    if variant not in cbam_results:
        continue
    h   = cbam_results[variant]['history']
    eps = range(1, len(h['train_loss']) + 1)
    c   = MODEL_COLORS[variant]

    axes[row,0].plot(eps, h['train_loss'], color=c, lw=2, label='Train')
    axes[row,0].plot(eps, h['val_loss'],   color=c, lw=2, ls='--', label='Val')
    axes[row,0].set_title(f'EfficientNet-{variant}+CBAM — Loss', fontweight='bold')
    axes[row,0].set_xlabel('Epoch'); axes[row,0].set_ylabel('Loss')
    axes[row,0].legend(); axes[row,0].spines[['top','right']].set_visible(False)

    axes[row,1].plot(eps, [x*100 for x in h['train_acc']], color=c, lw=2, label='Train')
    axes[row,1].plot(eps, [x*100 for x in h['val_acc']],   color=c, lw=2, ls='--', label='Val')
    axes[row,1].set_title(f'EfficientNet-{variant}+CBAM — Accuracy', fontweight='bold')
    axes[row,1].set_xlabel('Epoch'); axes[row,1].set_ylabel('Accuracy (%)')
    axes[row,1].legend(); axes[row,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_train_curves_cbam.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_train_curves_cbam.png')

## 🔢 Section 10: Confusion Matrices

In [ ]:
# ── Figure 3: Confusion matrices (Plain) ──────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('🔢 Confusion Matrices — Plain EfficientNet B0 → B7', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, variant in enumerate(ALL_VARIANTS):
    if variant not in plain_metrics:
        axes[idx].axis('off'); continue
    cm = plain_metrics[variant]['confusion_matrix']
    df_cm = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 11, 'weight': 'bold'})
    acc = plain_metrics[variant]['accuracy'] * 100
    axes[idx].set_title(f'EfficientNet-{variant}\nAcc: {acc:.2f}%', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('True', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_confusion_plain.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_confusion_plain.png')

In [ ]:
# ── Figure 4: Confusion matrices (CBAM) ──────────────────
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('🔢 Confusion Matrices — EfficientNet+CBAM B0 → B7', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, variant in enumerate(ALL_VARIANTS):
    if variant not in cbam_metrics:
        axes[idx].axis('off'); continue
    cm = cbam_metrics[variant]['confusion_matrix']
    df_cm = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    sns.heatmap(df_cm, annot=True, fmt='d', cmap='Purples', ax=axes[idx],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 11, 'weight': 'bold'})
    acc = cbam_metrics[variant]['accuracy'] * 100
    axes[idx].set_title(f'EfficientNet-{variant}+CBAM\nAcc: {acc:.2f}%', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('True', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_confusion_cbam.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_confusion_cbam.png')

## 📊 Section 11: Model Comparison — Plain vs CBAM

In [ ]:
# ── Figure 5: Accuracy comparison bar chart ───────────────
metric_keys   = ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'sensitivity', 'auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity', 'Sensitivity', 'AUC']

fig, axes = plt.subplots(2, 4, figsize=(24, 14))
fig.suptitle('🏆 Plain vs CBAM — All Metrics (B0 → B7)', fontsize=16, fontweight='bold')
axes = axes.flatten()

x     = np.arange(len(ALL_VARIANTS))
width = 0.35

for i, (key, label) in enumerate(zip(metric_keys, metric_labels)):
    plain_vals = [plain_metrics.get(v, {}).get(key, 0) * 100 for v in ALL_VARIANTS]
    cbam_vals  = [cbam_metrics.get(v, {}).get(key, 0) * 100 for v in ALL_VARIANTS]

    bars1 = axes[i].bar(x - width/2, plain_vals, width, label='Plain',
                        color=[MODEL_COLORS[v] for v in ALL_VARIANTS], alpha=0.85, edgecolor='white')
    bars2 = axes[i].bar(x + width/2, cbam_vals,  width, label='+CBAM',
                        color=[MODEL_COLORS[v] for v in ALL_VARIANTS], alpha=0.55, edgecolor='white',
                        hatch='//')

    axes[i].set_title(label, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('%' if key != 'auc' else 'Score')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(ALL_VARIANTS, fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].spines[['top','right']].set_visible(False)

    mn = min(min(plain_vals), min(cbam_vals))
    axes[i].set_ylim(max(0, mn*0.97), 101 if key != 'auc' else 1.02)

axes[-1].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_comparison_grouped_bars.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_comparison_grouped_bars.png')

In [ ]:
# ── Figure 6: Accuracy delta (CBAM gain) ──────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('📈 CBAM Accuracy Gain over Plain EfficientNet (B0 → B7)', fontsize=15, fontweight='bold')

deltas = []
for variant in ALL_VARIANTS:
    plain_acc = plain_metrics.get(variant, {}).get('accuracy', 0) * 100
    cbam_acc  = cbam_metrics.get(variant, {}).get('accuracy', 0) * 100
    deltas.append(cbam_acc - plain_acc)

colors = ['#2ECC71' if d >= 0 else '#E74C3C' for d in deltas]
bars   = ax.bar(ALL_VARIANTS, deltas, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='black', lw=1.2, ls='--')
ax.set_xlabel('EfficientNet Variant', fontsize=12)
ax.set_ylabel('Accuracy Δ (%)', fontsize=12)
ax.set_title('Green = CBAM improves, Red = CBAM hurts', fontsize=11, color='gray')

for bar, val in zip(bars, deltas):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{val:+.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_cbam_delta.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_cbam_delta.png')

In [ ]:
# ── Figure 7: Radar chart (Best Plain vs Best CBAM) ─────────
best_plain = max((v for v in plain_metrics), key=lambda v: plain_metrics[v]['accuracy'])
best_cbam  = max((v for v in cbam_metrics),  key=lambda v: cbam_metrics[v]['accuracy'])

radar_keys    = ['accuracy','precision','recall','f1','specificity','sensitivity']
radar_labels  = ['Accuracy','Precision','Recall','F1','Specificity','Sensitivity']
N      = len(radar_keys)
angles = [n/float(N)*2*np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(18, 8), subplot_kw=dict(polar=True))
fig.suptitle('🕸️ Radar Chart Comparison — All Variants', fontsize=15, fontweight='bold')

# Left: Plain
for variant in ALL_VARIANTS:
    if variant not in plain_metrics: continue
    vals  = [plain_metrics[variant][k]*100 for k in radar_keys] + [plain_metrics[variant][radar_keys[0]]*100]
    axes[0].plot(angles, vals, 'o-', lw=2, label=f'EfficientNet-{variant}', color=MODEL_COLORS[variant])
    axes[0].fill(angles, vals, alpha=0.05, color=MODEL_COLORS[variant])
axes[0].set_xticks(angles[:-1]); axes[0].set_xticklabels(radar_labels, size=11)
axes[0].set_ylim(80, 100); axes[0].set_title('Plain EfficientNet', fontweight='bold', pad=15)
axes[0].legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=9)

# Right: CBAM
for variant in ALL_VARIANTS:
    if variant not in cbam_metrics: continue
    vals  = [cbam_metrics[variant][k]*100 for k in radar_keys] + [cbam_metrics[variant][radar_keys[0]]*100]
    axes[1].plot(angles, vals, 'o-', lw=2, label=f'{variant}+CBAM', color=MODEL_COLORS[variant])
    axes[1].fill(angles, vals, alpha=0.05, color=MODEL_COLORS[variant])
axes[1].set_xticks(angles[:-1]); axes[1].set_xticklabels(radar_labels, size=11)
axes[1].set_ylim(80, 100); axes[1].set_title('EfficientNet + CBAM', fontweight='bold', pad=15)
axes[1].legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_radar_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_radar_comparison.png')

## 🌡️ Section 12: Per-Class Metrics Heatmaps

In [ ]:
# ── Figure 8: Sensitivity / Specificity heatmaps ──────────
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('🌡️ Per-Class Sensitivity & Specificity (%)', fontsize=16, fontweight='bold')

for row_idx, (results_dict, title_prefix, cmap) in enumerate([
    (plain_metrics, 'Plain', 'YlOrRd'),
    (cbam_metrics,  '+CBAM', 'YlGnBu'),
]):
    for col_idx, (metric_key, metric_title) in enumerate([
        ('per_class_sens', 'Sensitivity'),
        ('per_class_spec', 'Specificity'),
    ]):
        ax   = axes[row_idx][col_idx]
        data = np.array([[results_dict.get(v, {}).get(metric_key, [0]*NUM_CLASSES)[i]*100
                          for i in range(NUM_CLASSES)]
                         for v in ALL_VARIANTS])
        df_h = pd.DataFrame(data, index=ALL_VARIANTS, columns=CLASS_NAMES)
        sns.heatmap(df_h, annot=True, fmt='.1f', cmap=cmap,
                    vmin=75, vmax=100, ax=ax,
                    linewidths=0.5, linecolor='white',
                    annot_kws={'size':11, 'weight':'bold'})
        ax.set_title(f'{title_prefix} — {metric_title} (%)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Class'); ax.set_ylabel('Variant')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_perclass_heatmaps.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_perclass_heatmaps.png')

## 📉 Section 13: ROC-AUC Curves

In [ ]:
def plot_roc_for_group(results_dict, metrics_dict, title, filename, color_suffix=''):
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.suptitle(title, fontsize=15, fontweight='bold')
    axes = axes.flatten()

    for idx, variant in enumerate(ALL_VARIANTS):
        ax = axes[idx]
        if variant not in results_dict:
            ax.axis('off'); continue
        r      = results_dict[variant]
        y_bin  = label_binarize(r['test_labels'], classes=list(range(NUM_CLASSES)))
        probs  = r['test_probs']
        colors = ['#E74C3C','#2ECC71','#3498DB','#F39C12']
        for i, cls_name in enumerate(CLASS_NAMES):
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            roc_auc_val = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=2, color=colors[i], label=f'{cls_name} ({roc_auc_val:.3f})')
        ax.plot([0,1],[0,1],'k--',lw=1)
        ax.set_title(f'EfficientNet-{variant}\nMacro AUC: {metrics_dict.get(variant,{}).get("auc",0):.4f}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('FPR', fontsize=10); ax.set_ylabel('TPR', fontsize=10)
        ax.legend(loc='lower right', fontsize=8)
        ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {filename}')

plot_roc_for_group(plain_results, plain_metrics,
                   '📉 ROC-AUC — Plain EfficientNet B0 → B7',
                   'fig_roc_plain.png')

plot_roc_for_group(cbam_results, cbam_metrics,
                   '📉 ROC-AUC — EfficientNet+CBAM B0 → B7',
                   'fig_roc_cbam.png')

## 🔥 Section 14: Grad-CAM Visualization

In [ ]:
def get_gradcam_images(model, variant, num_samples=3):
    """Generate Grad-CAM for all 4 classes using the model's last conv block."""
    cfg      = EFFICIENTNET_CONFIGS[variant]
    img_size = cfg['img_size']
    model.eval()

    # Find target layer — last conv_pwl in the last stage
    target_layer = None
    for stage in reversed(list(model.blocks)):
        for blk in reversed(list(stage)):
            # Handle CBAMBlock wrapper
            inner = blk.block if hasattr(blk, 'block') else blk
            if hasattr(inner, 'conv_pwl'):
                target_layer = [inner.conv_pwl]
                break
        if target_layer:
            break
    if target_layer is None:
        print(f'  ⚠️ Could not find target layer for {variant}')
        return {}

    results = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_path  = os.path.join(TEST_DIR, cls_name)
        img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        samples   = random.sample(img_files, min(num_samples, len(img_files)))
        cam_list  = []
        with GradCAM(model=model, target_layers=target_layer) as cam:
            for img_file in samples:
                img_path  = os.path.join(cls_path, img_file)
                orig_img  = Image.open(img_path).convert('RGB').resize((img_size, img_size))
                img_array = np.array(orig_img) / 255.0
                tf  = get_transforms(img_size, 'val')
                inp = tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
                gs  = cam(input_tensor=inp, targets=[ClassifierOutputTarget(cls_idx)])
                ci  = show_cam_on_image(img_array.astype(np.float32), gs[0], use_rgb=True)
                cam_list.append((img_array, ci))
        results[cls_name] = cam_list
    return results

In [ ]:
# ── Figure 9: Grad-CAM best plain model ────────────────────
best_plain = max((v for v in plain_metrics), key=lambda v: plain_metrics[v]['accuracy'])
print(f'🏆 Best Plain: EfficientNet-{best_plain} ({plain_metrics[best_plain]["accuracy"]*100:.2f}%)')

cam_plain = get_gradcam_images(plain_results[best_plain]['model'], best_plain, num_samples=3)

fig, axes = plt.subplots(4, 6, figsize=(22, 14))
fig.suptitle(f'🔥 Grad-CAM — EfficientNet-{best_plain} (Best Plain)', fontsize=15, fontweight='bold')

for row, cls_name in enumerate(CLASS_NAMES):
    for s in range(3):
        if s >= len(cam_plain.get(cls_name, [])):
            axes[row, s*2].axis('off'); axes[row, s*2+1].axis('off'); continue
        orig, cam_img = cam_plain[cls_name][s]
        axes[row, s*2].imshow(orig); axes[row, s*2].axis('off')
        axes[row, s*2+1].imshow(cam_img); axes[row, s*2+1].axis('off')
        if s == 0:
            axes[row, 0].set_ylabel(cls_name, fontsize=11, fontweight='bold',
                                     color=CLASS_COLORS[cls_name],
                                     rotation=0, labelpad=75, va='center')
            axes[row, 0].set_title('Original', fontsize=9)
            axes[row, 1].set_title('Grad-CAM', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_gradcam_best_plain.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Saved: fig_gradcam_best_plain.png')

In [ ]:
# ── Figure 10: Grad-CAM best CBAM model ───────────────────
best_cbam = max((v for v in cbam_metrics), key=lambda v: cbam_metrics[v]['accuracy'])
print(f'🏆 Best CBAM: EfficientNet-{best_cbam}+CBAM ({cbam_metrics[best_cbam]["accuracy"]*100:.2f}%)')

cam_cbam = get_gradcam_images(cbam_results[best_cbam]['model'], best_cbam, num_samples=3)

fig, axes = plt.subplots(4, 6, figsize=(22, 14))
fig.suptitle(f'🔥 Grad-CAM — EfficientNet-{best_cbam}+CBAM (Best CBAM)', fontsize=15, fontweight='bold')

for row, cls_name in enumerate(CLASS_NAMES):
    for s in range(3):
        if s >= len(cam_cbam.get(cls_name, [])):
            axes[row, s*2].axis('off'); axes[row, s*2+1].axis('off'); continue
        orig, cam_img = cam_cbam[cls_name][s]
        axes[row, s*2].imshow(orig); axes[row, s*2].axis('off')
        axes[row, s*2+1].imshow(cam_img); axes[row, s*2+1].axis('off')
        if s == 0:
            axes[row, 0].set_ylabel(cls_name, fontsize=11, fontweight='bold',
                                     color=CLASS_COLORS[cls_name],
                                     rotation=0, labelpad=75, va='center')
            axes[row, 0].set_title('Original', fontsize=9)
            axes[row, 1].set_title('Grad-CAM', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_gradcam_best_cbam.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Saved: fig_gradcam_best_cbam.png')

## 📋 Section 15: Final Summary Table

In [ ]:
# ── Build summary DataFrame ───────────────────────────────
rows = []
for variant in ALL_VARIANTS:
    cfg = EFFICIENTNET_CONFIGS[variant]
    for mtype, mdict in [('Plain', plain_metrics), ('Plain+CBAM', cbam_metrics)]:
        if variant not in mdict: continue
        m = mdict[variant]
        rows.append({
            'Model'         : f'EfficientNet-{variant}' + ('+CBAM' if mtype=='Plain+CBAM' else ''),
            'Type'          : mtype,
            'Input'         : f'{cfg["img_size"]}²',
            'Params'        : cfg['params'],
            'Accuracy (%)'  : round(m['accuracy']*100, 2),
            'Precision (%)' : round(m['precision']*100, 2),
            'Recall (%)'    : round(m['recall']*100, 2),
            'F1-Score (%)'  : round(m['f1']*100, 2),
            'Specificity (%)':round(m['specificity']*100, 2),
            'Sensitivity (%)':round(m['sensitivity']*100, 2),
            'AUC'           : round(m['auc'], 4),
        })

# Add paper baseline
rows.append({
    'Model': 'Paper CNN (Bhandari et al.)', 'Type': 'Baseline',
    'Input': '180²', 'Params': '3.7M',
    'Accuracy (%)': 94.31, 'Precision (%)': '—', 'Recall (%)': '—',
    'F1-Score (%)': '—', 'Specificity (%)': '—', 'Sensitivity (%)': '—', 'AUC': '—'
})

df_summary = pd.DataFrame(rows)
df_summary.to_csv(f'{OUTPUT_DIR}/summary_results_b0_b7.csv', index=False)

print('='*120)
print('📋 FINAL RESULTS SUMMARY — EfficientNet B0→B7 Plain & CBAM')
print('='*120)
print(df_summary.to_string(index=False))
print('='*120)

best_plain_var = max((v for v in plain_metrics), key=lambda v: plain_metrics[v]['accuracy'])
best_cbam_var  = max((v for v in cbam_metrics),  key=lambda v: cbam_metrics[v]['accuracy'])
print(f'\n🥇 Best Plain : EfficientNet-{best_plain_var} — {plain_metrics[best_plain_var]["accuracy"]*100:.2f}%')
print(f'🥇 Best CBAM  : EfficientNet-{best_cbam_var}+CBAM — {cbam_metrics[best_cbam_var]["accuracy"]*100:.2f}%')
print(f'📄 Paper CNN  : 94.31%')
print(f'\n✅ Saved: summary_results_b0_b7.csv')

In [ ]:
# ── Figure 11: Visual summary table ───────────────────────
fig, ax = plt.subplots(figsize=(22, len(rows)*0.55 + 1.5))
ax.axis('off')

cols = ['Model','Type','Input','Params','Acc%','Prec%','Rec%','F1%','Spec%','Sens%','AUC']
table_data = []
for row in rows:
    table_data.append([
        row['Model'], row['Type'], row['Input'], row['Params'],
        str(row['Accuracy (%)']), str(row['Precision (%)']),
        str(row['Recall (%)']), str(row['F1-Score (%)']),
        str(row['Specificity (%)']), str(row['Sensitivity (%)']),
        str(row['AUC']),
    ])

tbl = ax.table(cellText=table_data, colLabels=cols, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.8)

# Header styling
for j in range(len(cols)):
    tbl[(0,j)].set_facecolor('#2C3E50')
    tbl[(0,j)].set_text_props(color='white', fontweight='bold')

# Row coloring — alternate plain/cbam/plain/cbam…
row_colors_plain = '#EBF5FB'
row_colors_cbam  = '#EBF8F2'
row_colors_base  = '#FDFEFE'

for i, row in enumerate(rows):
    color = row_colors_cbam if row['Type']=='Plain+CBAM' else (row_colors_base if row['Type']=='Baseline' else row_colors_plain)
    for j in range(len(cols)):
        tbl[(i+1, j)].set_facecolor(color)

ax.set_title('📊 Complete Results — EfficientNet B0→B7 Plain & CBAM', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_summary_table.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_summary_table.png')

In [ ]:
# ── List all outputs ─────────────────────────────────────
print('\n📁 ALL OUTPUT FILES:')
print('='*65)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'  📄 {f:<50} {size:6.1f} KB')
print('='*65)
print('\n🎉 NOTEBOOK COMPLETE!')